# 05 · EDA automatizado

> **Objetivo:** complementar el EDA manual con dos herramientas que
> condensan diagnosticos: **`skimpy`** (sobre Polars) para un resumen
> estadistico rapido y **`fg-data-profiling`** (fork de ydata-profiling)
> para un informe HTML completo sobre una muestra.

## Que vamos a hacer

1. Pasar `skim` sobre `ventas_validas.parquet` directamente en Polars.
2. Tomar una muestra reproducible de 100 000 filas (`seed=42`) y
   convertirla a pandas con `.to_pandas()` — **unico lugar del flujo
   U2 donde usamos pandas**.
3. Generar un informe HTML `minimal=True` en `reports/perfil_muestra.html`
   (gitignored).

## Advertencia

El perfil HTML sobre 1 M de filas tarda **minutos** y produce muchos
graficos; con una muestra de 100 000 filas y `minimal=True` tarda
~1-2 min, suficiente para la clase.

In [ ]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
if _cwd.name == "legacy":
    PROJECT_ROOT = _cwd.parent.parent
elif _cwd.name == "notebooks":
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from segmentacion.config import PROCESSED_DATA_DIR, REPORTS_DIR
from segmentacion.data.cargar import leer_parquet

PARQUET_VENTAS = PROCESSED_DATA_DIR / "ventas_validas.parquet"
assert PARQUET_VENTAS.exists(), "Ejecuta `02_limpieza.ipynb` primero."
ventas = leer_parquet(PARQUET_VENTAS)
print(f"ventas validas: {ventas.height:,}".replace(",", " "))

## 1. skimpy sobre Polars (sin convertir)

In [ ]:
from skimpy import skim

skim(ventas)

## 2. Muestra de 100 000 filas (seed=42)

La semilla fija garantiza que la clase y el video generen el **mismo**
subconjunto; si no, el perfil HTML cambia entre ejecuciones.

In [ ]:
muestra = ventas.sample(n=100_000, seed=42).to_pandas()
print(f"shape de la muestra pandas: {muestra.shape}")

## 3. Perfil HTML con `fg-data-profiling` (modulo `data_profiling`)

In [ ]:
from data_profiling import ProfileReport  # fg-data-profiling es un fork de ydata-profiling

perfil = ProfileReport(
    muestra,
    title="Perfil muestra — Online Retail II (U2)",
    minimal=True,
    progress_bar=False,
)
destino = REPORTS_DIR / "perfil_muestra.html"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
perfil.to_file(destino)
print(f"perfil escrito en: {destino}")
print(f"tamano: {destino.stat().st_size / 1e6:.1f} MB")

## Que mirar en el HTML

- **Overview:** porcentaje de nulos y duplicados por columna.
- **Variables:** histograma, cardinalidad, valores mas frecuentes.
- **Interactions / Correlations:** confirma Pearson y Spearman de
  `04_eda.ipynb` (Quantity vs Price).
- **Missing values:** matriz de patrones de nulos (en una muestra ya
  limpia, deberia estar practicamente vacia).

## Preguntas de reflexion

1. ¿Que agrega `skim` sobre lo que ya tienes en `04_eda.ipynb`? ¿Que
   quita?
2. ¿Por que `minimal=True`? ¿Que opciones encenderias si tuvieras una
   hora para esperar?
3. ¿Como sabrias si la muestra `n=100 000` representa bien a los 779 425
   validos? (Pista: compara `skim(ventas)` con `skim(muestra)`.)

## Fin de la Unidad 2

Siguiente parada: **`SPEC_procesamiento_U3.md`** — particion y fuga,
ingenieria de caracteristicas, pipelines y clustering. Los artefactos
`data/processed/{ventas_validas,cancelaciones,rfm_u2}.parquet` que
dejamos aqui son la entrada.